# 01 · Getting started with TMCBS

**TMCBS** (Transversal Multiple Code Block Simulator) builds *full circuit-level*
models of **distributed**, transversal fault-tolerant operations between
error-corrected nodes and decodes them.

The setting (paper Fig. 1): two quantum computers, `QC1` and `QC2`, each holding
logical code blocks, connected by an **entanglement station** that supplies noisy
**ebits** (shared Bell pairs). Using only transversal gates, measurements and
those ebits we can implement:

* a **non-local CNOT** between a control block on `QC1` and a target block on
  `QC2` (gate teleportation), and
* full **logical teleportation** of a code block from one node to the other.

This notebook builds a non-local CNOT, looks at its detector error model, decodes
it, and estimates a logical error rate. We work with the smallest codes so
everything runs in seconds on a laptop.

> Companion to *Transversal Fault Tolerant Distributed Quantum Computing
> Operations* (arXiv:2504.05611).


## Setup

Install the package with `pip install -e .` from the repository root, then run the
cell below to import it and pick a decoder.


In [7]:
# --- make the tmcbs package importable whether this notebook is launched from
# --- the package directory or from notebooks/ ------------------------------
import os, sys
for _cand in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), ".."))):
    if os.path.isdir(os.path.join(_cand, "tmcbs")) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import numpy as np
import tmcbs as t
print("tmcbs version", t.__version__)

# Decoder used throughout. "teser" (Tesseract) is the decoder behind the paper's
# main results. If the compiled tesseract_decoder wheel will not load on your
# machine, switch to a pure-Python fallback: "BP-OSD" or "LSD".
DECODER = "teser"


tmcbs version 0.1.0


## Codes

TMCBS ships a registry of the codes studied in the paper: rotated **surface
codes** `[[d², 1, d]]` and bivariate-bicycle (**BB**) qLDPC codes. Each lookup
returns a lightweight `Code` handle carrying the metadata the rest of the library
needs (`n`, `k`, `d`, family) plus the object the circuit builders expect.


In [8]:
print(t.list_codes())

sc = t.surface_code(3)                 # [[9,1,3]]
bb = t.bb_code("[[18,4,4]] BB")        # smallest bivariate-bicycle code

for c in (sc, bb):
    print(f"{c.name:18s} family={c.family:7s} n={c.n:3d} k={c.k:2d} d={c.d:2d} "
          f"encoding rate k/n={c.k/c.n:.3f}")


['[[9,1,3]] SC', '[[25,1,5]] SC', '[[49,1,7]] SC', '[[81,1,9]] SC', '[[121,1,11]] SC', '[[18,4,4]] BB', '[[36,4,6]] BB', '[[54,4,8]] BB', '[[90,8,10]] BB', '[[144,12,12]] BB']
[[9,1,3]] SC       family=surface n=  9 k= 1 d= 3 encoding rate k/n=0.111
[[18,4,4]] BB      family=bb      n= 18 k= 4 d= 4 encoding rate k/n=0.222


A surface code encodes a single logical qubit (`k = 1`); the BB codes encode
**many** logical qubits per block (`k = 4` here), which is what later lets a
*single* transversal CNOT act on `k` logical qubit pairs at once.


## Build a non-local CNOT

`non_local_cnot(code, p, p_ebit)` returns a fully-noisy `stim.Circuit`:

* `p` — the physical error rate (depolarising on 1- and 2-qubit gates,
  reset/measurement flips, idle noise),
* `p_ebit` — the ebit infidelity (2-qubit depolarising on each Bell pair).

The circuit already declares **detectors** (parities expected to be deterministic
in the absence of noise) and **logical observables** (the quantities we must
decode correctly).


In [9]:
circ = t.non_local_cnot(sc, p=1e-3, p_ebit=1e-3)

print("qubits      :", circ.num_qubits)
print("detectors   :", circ.num_detectors)     # matches Table III "rows" (64 for [[9,1,3]])
print("observables :", circ.num_observables)

# Peek at the first handful of instructions.
print("\n--- first 12 instructions ---")
for instr in circ[:12]:
    print(" ", instr)


qubits      : 52
detectors   : 64
observables : 2

--- first 12 instructions ---
  QUBIT_COORDS(1, 1) 0
  R 0
  X_ERROR(0.001) 0
  QUBIT_COORDS(3, 1) 1
  R 1
  X_ERROR(0.001) 1
  QUBIT_COORDS(5, 1) 2
  R 2
  X_ERROR(0.001) 2
  QUBIT_COORDS(1, 3) 3
  R 3
  X_ERROR(0.001) 3


## Detector error model, sampling and decoding

A **detector error model** (DEM) lists every independent error mechanism, which
detectors it flips, and which logical observables it flips. TMCBS converts the DEM
into a parity-check matrix and decodes the sampled syndromes.

Let's sample a few shots and look at one syndrome.


In [10]:
dem = circ.detector_error_model()
print("DEM error mechanisms:", dem.num_errors)

sampler = circ.compile_detector_sampler()
det, obs = sampler.sample(5, separate_observables=True)
print("syndrome weights (number of flipped detectors) over 5 shots:",
      det.sum(axis=1))
print("observable flips per shot:", obs.sum(axis=1))


DEM error mechanisms: 245
syndrome weights (number of flipped detectors) over 5 shots: [0 0 0 0 2]
observable flips per shot: [0 0 0 0 0]


## Estimate the logical error rate

`estimate_ler` keeps drawing shots until it has accumulated `target_errors`
logical failures — so the *relative* uncertainty is roughly constant even when the
LER is tiny — then reports the rate with **Bayes-factor binomial error bars** (the
same `sinter.fit_binomial`, factor 1000, used for the paper's figures).

A logical failure means *any* logical observable was decoded incorrectly (the
paper's whole-operation convention).


In [11]:
res = t.estimate_ler(circ, p=1e-3, d=sc.d, decoder=DECODER, target_errors=40)
print(res)
print(f"  errors / shots : {res.errors} / {res.shots}")
print(f"  Bayes CI       : [{res.ci_low:.2e}, {res.ci_high:.2e}]")
print(f"  wall time      : {res.seconds:.1f} s")


LER = 4.193e-03  [2.793e-03, 5.996e-03]  (95 errors / 22656 shots, teser)
  errors / shots : 95 / 22656
  Bayes CI       : [2.79e-03, 6.00e-03]
  wall time      : 0.8 s


### Scaling with the physical error rate

Below threshold, lowering the physical error rate lowers the logical error rate.
`target_errors` is kept modest here for speed; raise it for smoother numbers.


In [12]:
print(f"{'p':>8}   {'LER':>10}   {'[CI low, CI high]':>24}")
for p in (1e-2, 5e-3, 3e-3):
    c = t.non_local_cnot(sc, p=p, p_ebit=p)
    r = t.estimate_ler(c, p=p, d=sc.d, decoder=DECODER, target_errors=40)
    print(f"{p:>8.0e}   {r.ler:>10.2e}   [{r.ci_low:.2e}, {r.ci_high:.2e}]")


       p          LER          [CI low, CI high]
   1e-02     2.27e-01   [1.40e-01, 3.32e-01]
   5e-03     8.85e-02   [4.83e-02, 1.44e-01]
   3e-03     3.12e-02   [1.99e-02, 4.62e-02]


## Where to go next

* **`02_building_circuits.ipynb`** — the code registry and the circuit-builder
  vocabulary, reproducing the gate/DEM counts of Table III and extracting the
  parity-check matrix for your own decoder.
* **`03_reproduce_paper.ipynb`** — scaled-down reproductions of the paper's key
  figures (ebit noise, ebit decoherence, Pauli-string weight).
* For publication-scale statistics, drive the same building blocks across a
  cluster with the MPI runner (`python -m tmcbs.runEbitExperimentMPI`); ready-made
  sweeps live in `scripts/`.
